# Evaluate Optuna Hyperparameter Tuning

In [ ]:
import ipywidgets as widgets
import matplotlib.pyplot as plt
import numpy as np
import optuna
import optuna.visualization as vis
import seaborn as sns
from IPython.display import clear_output, display
from ipywidgets import interact


In [ ]:
study = optuna.load_study(
    study_name="flowmatching_20251029_2_hyperparameters_dev",
    storage="sqlite:///optuna_study.db"
)

print(f"Best value: {study.best_value:.06f}")
print("Best params:", study.best_params)
df = study.trials_dataframe()
df

### Filter Best Trials for specific parameters

In [ ]:
@interact(model_size=["S", "M", "L"])
def show_best(model_size):
    best = (
        df[df["params_MODEL_SIZE"] == model_size]
        .sort_values("value")
        .head(3)
    )
    display(best)

In [ ]:
@interact(model_size=["S", "M", "L"])
def show_shortest(model_size):
    shortest = (
        df[df["params_MODEL_SIZE"] == model_size]
        .sort_values("duration")
        .head(3)
    )
    display(shortest)

In [ ]:
mask = df["params_MODEL_SIZE"] == "M"
subset = df[mask].sort_values("value", ascending=True)
best_M = subset.iloc[0]
print("Best MODEL_SIZE=M trial:")
print(best_M[["number", "value"] + [c for c in df.columns if c.startswith("params_")]])

In [ ]:
sns.boxplot(data=df, x="params_MODEL_SIZE", y="value")
plt.title("CRPS by Model Size")
plt.ylabel("CRPS (lower is better)")
plt.show()

In [ ]:
best_per_size = (
    df.loc[df.groupby("params_MODEL_SIZE")["value"].idxmin(), ["params_MODEL_SIZE", "number", "value"]]
    .rename(columns={"value": "best_CRPS", "number": "trial_number"})
    .reset_index(drop=True)
)

print(best_per_size)

In [ ]:
pivot = df.pivot_table(
    index="params_lr",
    columns="params_step_size",
    values="value",
    aggfunc="min"
)

sns.heatmap(pivot, annot=True, fmt=".3f", cmap="cividis_r", cbar_kws={'label': 'CRPS'})
plt.title("CRPS (lower better) — lr vs step_size")
plt.xlabel("step_size")
plt.ylabel("lr")
plt.show()

In [ ]:
# vis.plot_param_importances(study)
# vis.plot_slice(study, params=["lr", "weight_decay"])

In [ ]:
@interact(model_size=["S", "M", "L"])
def explore_model_size(model_size):
    subset = df[df["params_MODEL_SIZE"] == model_size].copy()
    subset = subset.sort_values("value").reset_index(drop=True)

    display(subset[[
        "number", "value", "params_batch_size_train", "params_lr",
        "params_weight_decay", "params_max_lr", "params_min_lr",
        "params_halfcosine_steps", "params_step_size", "params_warmup_steps",
        "params_method"
    ]])

    plt.figure(figsize=(6, 3))
    sns.barplot(data=subset, x="number", y="value", hue="params_method")
    plt.title(f"CRPS per Trial for MODEL_SIZE={model_size}")
    plt.ylabel("CRPS (lower is better)")
    plt.show()

In [ ]:
# Round helper columns (if not already done)
df["params_min_lr_round"] = df["params_min_lr"].round(7)
df["params_max_lr_round"] = df["params_max_lr"].round(1)

# --- Define dropdown widgets ---
model_size_dd = widgets.Dropdown(options=sorted(df["params_MODEL_SIZE"].unique()), description="Model size")
step_size_dd = widgets.Dropdown(description="Step size")
method_dd = widgets.Dropdown(description="Method")
batch_size_dd = widgets.Dropdown(description="Batch size")
lr_dd = widgets.Dropdown(description="LR")
weight_decay_dd = widgets.Dropdown(description="Weight decay")
warmup_dd = widgets.Dropdown(description="Warmup steps")
halfcosine_dd = widgets.Dropdown(description="Halfcosine steps")
min_lr_dd = widgets.Dropdown(description="min LR")
max_lr_dd = widgets.Dropdown(description="max LR")

out = widgets.Output()

# --- Define update logic ---
def update_dropdowns(change=None):
    filtered = df.copy()

    # Filter step by step depending on what's selected
    if model_size_dd.value:
        filtered = filtered[filtered["params_MODEL_SIZE"] == model_size_dd.value]
    step_size_dd.options = sorted(filtered["params_step_size"].unique())

    if step_size_dd.value in step_size_dd.options:
        filtered = filtered[filtered["params_step_size"] == step_size_dd.value]
    method_dd.options = sorted(filtered["params_method"].unique())

    if method_dd.value in method_dd.options:
        filtered = filtered[filtered["params_method"] == method_dd.value]
    batch_size_dd.options = sorted(filtered["params_batch_size_train"].unique())

    if batch_size_dd.value in batch_size_dd.options:
        filtered = filtered[filtered["params_batch_size_train"] == batch_size_dd.value]
    lr_dd.options = sorted(filtered["params_lr"].unique())

    if lr_dd.value in lr_dd.options:
        filtered = filtered[filtered["params_lr"] == lr_dd.value]
    weight_decay_dd.options = sorted(filtered["params_weight_decay"].unique())

    if weight_decay_dd.value in weight_decay_dd.options:
        filtered = filtered[filtered["params_weight_decay"] == weight_decay_dd.value]
    warmup_dd.options = sorted(filtered["params_warmup_steps"].unique())
    halfcosine_dd.options = sorted(filtered["params_halfcosine_steps"].unique())
    min_lr_dd.options = sorted(filtered["params_min_lr_round"].unique())
    max_lr_dd.options = sorted(filtered["params_max_lr_round"].unique())

def show_results(change=None):
    filtered = df.copy()
    conditions = [
        ("params_MODEL_SIZE", model_size_dd.value),
        ("params_step_size", step_size_dd.value),
        ("params_method", method_dd.value),
        ("params_batch_size_train", batch_size_dd.value),
        ("params_lr", lr_dd.value),
        ("params_weight_decay", weight_decay_dd.value),
        ("params_warmup_steps", warmup_dd.value),
        ("params_halfcosine_steps", halfcosine_dd.value),
        ("params_min_lr_round", min_lr_dd.value),
        ("params_max_lr_round", max_lr_dd.value),
    ]
    for col, val in conditions:
        if val is not None:
            filtered = filtered[filtered[col] == val]

    with out:
        clear_output()
        if filtered.empty:
            display("⚠️ No trials match these parameters.")
        else:
            display(filtered.sort_values("value").head(10))

# --- Link updates ---
for dd in [model_size_dd, step_size_dd, method_dd, batch_size_dd, lr_dd,
           weight_decay_dd, warmup_dd, halfcosine_dd, min_lr_dd, max_lr_dd]:
    dd.observe(update_dropdowns, names="value")
    dd.observe(show_results, names="value")

# --- Initialize ---
update_dropdowns()
display(widgets.VBox([
    widgets.HBox([model_size_dd, step_size_dd, method_dd]),
    widgets.HBox([batch_size_dd, lr_dd, weight_decay_dd]),
    widgets.HBox([warmup_dd, halfcosine_dd]),
    widgets.HBox([min_lr_dd, max_lr_dd]),
    out
]))

### Even more (unnecessary) evaluation

In [ ]:
@interact(step_size=[0.05, 0.1, 0.2])
def show_best(step_size):
    best = (
        df[df["params_step_size"] == step_size]
        .sort_values("value")
        .head(3)
    )
    display(best)

In [ ]:
@interact(method=["midpoint", "euler"])
def show_best(method):
    best = (
        df[df["params_method"] == method]
        .sort_values("value")
        .head(3)
    )
    display(best)

In [ ]:
@interact(batch_size_train=[32, 64, 128])
def show_best(batch_size_train):
    best = (
        df[df["params_batch_size_train"] == batch_size_train]
        .sort_values("value")
        .head(3)
    )
    display(best)

In [ ]:
@interact(lr=[1e-4, 3e-4, 1e-3, 3e-3])
def show_best(lr):
    best = (
        df[df["params_lr"] == lr]
        .sort_values("value")
        .head(3)
    )
    display(best)

In [ ]:
@interact(weight_decay=[0.0, 0.01, 0.1])
def show_best(weight_decay):
    best = (
        df[df["params_weight_decay"] == weight_decay]
        .sort_values("value")
        .head(3)
    )
    display(best)

In [ ]:
@interact(warmup_steps=np.arange(500, 5001, 500))
def show_best(warmup_steps):
    best = (
        df[df["params_warmup_steps"] == warmup_steps]
        .sort_values("value")
        .head(3)
    )
    display(best)

In [ ]:
@interact(halfcosine_steps=np.arange(10000, 100001, 10000))
def show_best(halfcosine_steps):
    best = (
        df[df["params_halfcosine_steps"] == halfcosine_steps]
        .sort_values("value")
        .head(3)
    )
    display(best)

In [ ]:
df["params_min_lr_round"] = df["params_min_lr"].round(7)

min_lr_list = [round(v * scale, 8) for scale in [1e-7, 1e-6, 1e-5] for v in np.arange(1, 10, 2)]

@interact(min_lr=min_lr_list)
def show_best(min_lr):
    best = (
        df[df["params_min_lr_round"] == round(min_lr, 7)]
        .sort_values("value")
        .head(3)
    )
    display(best)

In [ ]:
df["params_max_lr_round"] = df["params_max_lr"].round(1)

max_lr_rate = [round(v, 2) for v in np.arange(0.1, 1.01, 0.1)]
@interact(max_lr=max_lr_rate)
def show_best(max_lr):
    best = (
        df[df["params_max_lr_round"] == round(max_lr, 1)]
        .sort_values("value")
        .head(3)
    )
    display(best)

In [ ]:
# Precompute rounded helper columns for matching
df["params_min_lr_round"] = df["params_min_lr"].round(7)
df["params_max_lr_round"] = df["params_max_lr"].round(1)

# Define slider and dropdown options
model_sizes = ["S", "M", "L"]
step_sizes = [0.05, 0.1, 0.2]
methods = ["midpoint", "euler"]
batch_sizes = [32, 64, 128]
lrs = [1e-4, 3e-4, 1e-3, 3e-3]
weight_decays = [0.0, 0.01, 0.1]
warmup_steps = np.arange(500, 5001, 500)
halfcosine_steps = np.arange(10000, 100001, 10000)
min_lr_list = [round(v * scale, 8) for scale in [1e-7, 1e-6, 1e-5] for v in np.arange(1, 10, 2)]
max_lr_list = [round(v, 2) for v in np.arange(0.1, 1.01, 0.1)]

@interact(
    model_size=model_sizes,
    step_size=step_sizes,
    method=methods,
    batch_size_train=batch_sizes,
    lr=lrs,
    weight_decay=weight_decays,
    warmup_steps=warmup_steps,
    halfcosine_steps=halfcosine_steps,
    # min_lr=min_lr_list,
    # max_lr=max_lr_list,
)
def show_best_all(
    model_size, step_size, method, batch_size_train,
    lr, weight_decay, warmup_steps, halfcosine_steps,
    ): # min_lr, max_lr):

    # Filter progressively (only active constraints)
    filtered = df.copy()
    filtered = filtered[
        (filtered["params_MODEL_SIZE"] == model_size)
        & (filtered["params_step_size"] == step_size)
        & (filtered["params_method"] == method)
        & (filtered["params_batch_size_train"] == batch_size_train)
        & (filtered["params_lr"] == lr)
        & (filtered["params_weight_decay"] == weight_decay)
        & (filtered["params_warmup_steps"] == warmup_steps)
        & (filtered["params_halfcosine_steps"] == halfcosine_steps)
        # & (filtered["params_min_lr_round"] == round(min_lr, 7))
        # & (filtered["params_max_lr_round"] == round(max_lr, 1))
    ]

    if len(filtered) == 0:
        display("⚠️ No matching trials found for this combination.")
    else:
        display(filtered.sort_values("value").head(5))